In [ ]:
!pip install transformers pypdf gradio groq -q

In [ ]:
import gradio as gr
from transformers import pipeline
from groq import Groq
from google.colab import userdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pypdf import PdfReader

# ---- Setup ----
sentiment_analyzer = pipeline("sentiment-analysis")
client = Groq(api_key=userdata.get('GROK_API'))

vectorizer = None
chunks = []

# ---- Sentiment Function ----
def analyze_sentiment(text):
    if not text.strip():
        return "ERROR: No input detected"
    result = sentiment_analyzer(text)[0]
    label = result['label']
    score = round(result['score'] * 100, 1)
    symbol = "[+]" if label == "POSITIVE" else "[-]"
    return f"{symbol} RESULT: {label}\n>> Confidence: {score}%\n>> Analysis complete."

# ---- PDF Functions ----
def load_pdf(file):
    global vectorizer, chunks
    if file is None:
        return "ERROR: No file received"
    reader = PdfReader(file)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    words = text.split()
    chunks = [" ".join(words[i:i+500]) for i in range(0, len(words), 500)]
    vectorizer = TfidfVectorizer()
    vectorizer.fit_transform(chunks)
    return f"[+] LOADED SUCCESSFULLY\n>> Pages: {len(reader.pages)}\n>> Chunks: {len(chunks)}\n>> Status: READY"

def ask_pdf(question):
    global vectorizer, chunks
    if not chunks:
        return "ERROR: No document loaded. Upload a PDF first."
    if not question.strip():
        return "ERROR: Empty query detected."

def summarize_text(text):
    if not text.strip():
        return "ERROR: No input detected"

def explain_code(code):
    if not code.strip():
        return "<span style='color:#00ff41'>ERROR: No code detected</span>"

    prompt = f"""Explain this code.
Format your response exactly like this for each part:
CODE: [the code line or block]
EXPLANATION: [what it does]

Code:
{code}"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.choices[0].message.content
    html = "<div style='font-family:Courier New; background:#0a0a0a; padding:10px'>"

    for line in raw.split('\n'):
        if line.startswith('CODE:'):
            html += f"<div style='color:#ff6600; margin:4px 0'>{line}</div>"
        elif line.startswith('EXPLANATION:'):
            html += f"<div style='color:#00ff41; margin:4px 0'>{line}</div>"
        else:
            html += f"<div style='color:#00ff41; margin:2px 0'>{line}</div>"

    html += "</div>"
    return html

    prompt = f"""Explain what this code does in simple plain English.
Break it down line by line if needed. Assume the reader is a beginner.

Code:
{code}

Explanation:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return f"[+] EXPLANATION GENERATED\n>> {response.choices[0].message.content}"

    prompt = f"""Summarize the following text in a clear and concise way.
Keep it under 5 bullet points. Be direct.

Text: {text}

Summary:"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return f"[+] SUMMARY GENERATED\n>> {response.choices[0].message.content}"
    question_vector = vectorizer.transform([question])
    chunk_vectors = vectorizer.transform(chunks)
    similarities = cosine_similarity(question_vector, chunk_vectors)
    best_chunk = chunks[np.argmax(similarities)]
    prompt = f"""Answer this question using only the context below.
If the answer isn't in the context, say "NOT FOUND IN DOCUMENT."

Context: {best_chunk}

Question: {question}

Answer:"""
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return f"[+] QUERY PROCESSED\n>> {response.choices[0].message.content}"

# ---- Custom CSS ----
css = """
* { font-family: 'Courier New', monospace !important; }

body, .gradio-container {
    background-color: #0a0a0a !important;
    color: #00ff41 !important;
}

/* Gradio 5 tab selectors */
.tab-nav > div > button {
    background-color: #003b00 !important;
    border: 1px solid #00ff41 !important;
    color: #00ff41 !important;
    padding: 8px 16px !important;
    margin: 3px !important;
    border-radius: 3px !important;
}

.tab-nav > div > button:hover {
    background-color: #00ff41 !important;
    color: #0a0a0a !important;
}

button[aria-selected="true"] {
    background-color: #00ff41 !important;
    color: #0a0a0a !important;
    font-weight: bold !important;
}

/* Hide scrollbar */
::-webkit-scrollbar { display: none; }
* { scrollbar-width: none; -ms-overflow-style: none; }

/* Inputs */
textarea, input, .gr-box {
    background-color: #0d0d0d !important;
    border: 1px solid #00ff41 !important;
    color: #00ff41 !important;
}

/* Buttons */
.gr-button-primary {
    background-color: #003b00 !important;
    border: 1px solid #00ff41 !important;
    color: #00ff41 !important;
}
.gr-button-primary:hover {
    background-color: #00ff41 !important;
    color: #0a0a0a !important;
}

label, .gr-block-label { color: #00ff41 !important; }
.gr-panel { background-color: #0a0a0a !important; border: 1px solid #003b00 !important; }
"""

# ---- UI ----
with gr.Blocks(css=css, title="ByteMind") as app:

    gr.Markdown("""
```
██████╗ ██╗   ██╗████████╗███████╗███╗   ███╗██╗███╗   ██╗██████╗
██╔══██╗╚██╗ ██╔╝╚══██╔══╝██╔════╝████╗ ████║██║████╗  ██║██╔══██╗
██████╔╝ ╚████╔╝    ██║   █████╗  ██╔████╔██║██║██╔██╗ ██║██║  ██║
██╔══██╗  ╚██╔╝     ██║   ██╔══╝  ██║╚██╔╝██║██║██║╚██╗██║██║  ██║
██████╔╝   ██║      ██║   ███████╗██║ ╚═╝ ██║██║██║ ╚████║██████╔╝
╚═════╝    ╚═╝      ╚═╝   ╚══════╝╚═╝     ╚═╝╚═╝╚═╝  ╚═══╝╚═════╝
```
**[ AI at its Core ]** — v1.0.0 | Status: ONLINE
    """)

    with gr.Tabs():
        with gr.Tab("[ 01 ] SENTIMENT ANALYSIS"):
            gr.Markdown("### > Analyze emotional polarity of any text input")
            sentiment_input = gr.Textbox(label="INPUT //", placeholder="Enter text string...", lines=3)
            sentiment_button = gr.Button(">> RUN ANALYSIS", variant="primary")
            sentiment_output = gr.Textbox(label="OUTPUT //", lines=4)
            sentiment_button.click(fn=analyze_sentiment, inputs=sentiment_input, outputs=sentiment_output)

        with gr.Tab("[ 02 ] PDF QUERY ENGINE"):
            gr.Markdown("### > Upload document and query it with natural language")
            pdf_upload = gr.File(label="LOAD DOCUMENT //", file_types=[".pdf"])
            upload_status = gr.Textbox(label="SYSTEM LOG //", lines=4)
            pdf_upload.change(fn=load_pdf, inputs=pdf_upload, outputs=upload_status)
            pdf_question = gr.Textbox(label="QUERY //", placeholder="Enter your query...", lines=2)
            pdf_button = gr.Button(">> EXECUTE QUERY", variant="primary")
            pdf_answer = gr.Textbox(label="RESPONSE //", lines=6)
            pdf_button.click(fn=ask_pdf, inputs=pdf_question, outputs=pdf_answer)

        with gr.Tab("[ 03 ] TEXT SUMMARIZER"):
            gr.Markdown("### > Paste any text and get a concise summary")
            summary_input = gr.Textbox(
                label="INPUT //",
                placeholder="Paste any article, paragraph or long text...",
                lines=8
            )
            summary_button = gr.Button(">> SUMMARIZE", variant="primary")
            summary_output = gr.Textbox(label="SUMMARY //", lines=6)
            summary_button.click(
                fn=summarize_text,
                inputs=summary_input,
                outputs=summary_output
            )

        with gr.Tab("[ 04 ] CODE EXPLAINER"):
            gr.Markdown("### > Paste any code and AI will explain it")
            code_input = gr.Code(
                label="INPUT //",
                language="python",
                lines=10
            )
            code_button = gr.Button(">> EXPLAIN CODE", variant="primary")
            gr.Markdown("**EXPLANATION //**")
            code_output = gr.HTML()
            code_button.click(
                fn=explain_code,
                inputs=code_input,
                outputs=code_output
            )

app.launch(share=True)